In [ ]:
import pandas as pd

# view the columns and number of rows in test_dataset.parquet
df = pd.read_parquet("test_dataset.parquet")
print("Columns:", list(df.columns))
print("Number of rows:", len(df))

In [ ]:
# get the first row of df
df.iloc[1]

In [ ]:
# Select rows where the 'hash' column matches the specific value
matching_rows = df.loc[df['hash'] == "0x15190f93d394645bc7272ee884fa86f98f06d27e08a6a45cf4cffe0473199460"]

# If you expect only one row and want it as a Series (like your example)
# You can take the first result using .iloc[0]
if not matching_rows.empty:
    target_row = matching_rows.iloc[0]
    print(target_row)
else:
    print("No row found with that hash.")

In [ ]:
# let's store a version of df to a csv file with only hash and failure_invariant columns
df[['hash', 'failure_invariant']].to_csv("test_dataset_hash_invariant.csv", index=True)


In [ ]:
# Flexible matching of clustered invariants to the 20k Parquet dataset.
# - Parses cluster headers like: --- Cluster 17 (13 items) ---
# - Normalizes invariants whitespace-insensitively
# - Extracts predicates from: require(...), assert(...), and if(...){revert...}
# - Attempts simple logical inversion for if(...) revert patterns (>,<,>=,<=,==,!=)
# Outputs:
#   invariant_failure_counts.csv  (cluster_id, invariant, inv_key, tx_fail_count)
#   cluster_tx_totals.csv         (cluster_id, cluster_tx_fail_total)

import re
from pathlib import Path
import pandas as pd

CLUSTERS_TXT = "dbscan_Raven.txt"
PARQUET_FP   = "test_dataset.parquet"

# ---------- normalization ----------
def normalize_key(s: str) -> str:
    if s is None:
        return ""
    s = s.strip()
    s = re.sub(r"^\s*[-–—•]\s*", "", s)          # leading bullet
    s = s.strip('"\'' + "`")                     # surrounding quotes/backticks
    s = s.rstrip(",;.")                          # trailing punctuation
    s = s.lower()
    s = re.sub(r"\s+", "", s)                    # drop all whitespace
    return s

# ---------- cluster parser ----------
def parse_clusters(path: str) -> pd.DataFrame:
    clusters = []
    current = None
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.rstrip("\n")
            m = re.match(r"^\s*---\s*cluster\s+(\d+)\s*\((\d+)\s*items?\)\s*---\s*$", line, flags=re.IGNORECASE)
            if m:
                current = int(m.group(1))
                continue
            if current is None or not line.strip() or line.strip().startswith("---"):
                continue
            inv_raw = re.sub(r"^\s*-\s*", "", line).strip()
            if not inv_raw:
                continue
            clusters.append({"cluster_id": current, "invariant": inv_raw, "inv_key": normalize_key(inv_raw)})
    df = pd.DataFrame(clusters)
    if df.empty:
        raise ValueError("No invariants parsed from clusters file. Check formatting.")
    return df

# ---------- predicate extraction helpers ----------
_bin_ops = [">=", "<=", ">", "<", "==", "!="]
_invert = {">": "<=", "<": ">=", ">=": "<", "<=": ">", "==": "!=", "!=": "=="}

def try_invert_simple(expr: str) -> str:
    # Try to invert a simple binary comparison A op B -> A inv(op) B
    # Heuristic: find the *first* binary op occurrence
    for op in _bin_ops:
        parts = expr.split(op)
        if len(parts) == 2:
            left, right = parts
            inv_op = _invert.get(op)
            if inv_op:
                return f"{left}{inv_op}{right}"
    return ""

def extract_predicate_variants(s: str) -> list:
    """Return a list of candidate invariant strings (not yet normalized)."""
    out = []
    if not s:
        return out

    ss = s.strip()

    # 1) Raw
    out.append(ss)

    # 2) require(...) / assert(...)
    for kw in ("require", "assert"):
        m = re.search(rf"{kw}\s*\((.*)\)", ss, flags=re.IGNORECASE|re.DOTALL)
        if m:
            inside = m.group(1).strip()
            out.append(inside)
            # Split at first comma to drop message if present
            parts = inside.split(",", 1)
            out.append(parts[0].strip())

    # 3) if (COND) revert ...   (we try original COND, !-stripped, and inverted)
    m_if = re.search(r"if\s*\((.*)\)\s*{{0,1}\s*revert", ss, flags=re.IGNORECASE)
    if m_if:
        cond = m_if.group(1).strip()
        out.append(cond)
        # strip leading negation !(...)
        m_not = re.match(r"!\s*\((.*)\)\s*$", cond)
        if m_not:
            out.append(m_not.group(1).strip())
        # try invert simple A op B
        inv = try_invert_simple(cond.replace(" ", ""))
        if inv:
            out.append(inv)

    # 4) Also handle patterns like "revert When(!(cond));" (custom errors); extract cond inside !(...)
    m_custom = re.search(r"revert\s+[A-Za-z_]\w*\s*\((.*)\)", ss)
    if m_custom:
        inner = m_custom.group(1).strip()
        out.append(inner)
        m_not2 = re.match(r"!\s*\((.*)\)\s*$", inner)
        if m_not2:
            out.append(m_not2.group(1).strip())

    # Deduplicate while preserving order
    seen = set()
    uniq = []
    for v in out:
        if v not in seen:
            seen.add(v)
            uniq.append(v)
    return uniq

# ---------- main ----------
clusters_df = parse_clusters(CLUSTERS_TXT)

# Load Parquet (pandas with pyarrow/fastparquet)
dataset_df = pd.read_parquet(PARQUET_FP)

# Pick invariant column (prefer 'failure_invariant')
inv_col = None
if "failure_invariant" in dataset_df.columns:
    inv_col = "failure_invariant"
else:
    cand = [c for c in dataset_df.columns if any(k in c.lower() for k in ["invariant","predicate"])]
    if not cand:
        raise KeyError(f"'failure_invariant' column not found. Available: {list(dataset_df.columns)}")
    inv_col = cand[0]
dataset_df = dataset_df[dataset_df[inv_col].notna()].copy()

# Build candidate keys per row (explode)
dataset_df = dataset_df.reset_index(drop=True).rename(columns={inv_col: "failure_invariant"})
dataset_df["row_id"] = dataset_df.index

candidates = []
for row in dataset_df.itertuples(index=False):
    preds = extract_predicate_variants(getattr(row, "failure_invariant"))
    for p in preds:
        key = normalize_key(p)
        if key:  # avoid empty keys
            candidates.append((row.row_id, key))

cand_df = pd.DataFrame(candidates, columns=["row_id", "inv_key"]).drop_duplicates()

# Counts per candidate key (unique rows)
counts = cand_df.groupby("inv_key")["row_id"].nunique().rename("tx_fail_count")

# Join to cluster invariants
result = clusters_df.merge(counts, how="left", on="inv_key")
result["tx_fail_count"] = result["tx_fail_count"].fillna(0).astype(int)

# Coverage: how many tx rows are explained by cluster keys?
covered_rows = cand_df[cand_df["inv_key"].isin(result["inv_key"])].row_id.nunique()
n_rows = len(dataset_df)
coverage_pct = 100.0 * covered_rows / max(n_rows, 1)

# Cluster totals
cluster_totals = (result.groupby("cluster_id")["tx_fail_count"]
                  .sum()
                  .reset_index()
                  .rename(columns={"tx_fail_count": "cluster_tx_fail_total"})
                  .sort_values("cluster_tx_fail_total", ascending=False))

# Save
out_inv = "invariant_failure_counts.csv"
out_cluster = "cluster_tx_totals.csv"
result.to_csv(out_inv, index=False)
cluster_totals.to_csv(out_cluster, index=False)

# Summary
print(f"Dataset rows: {n_rows}")
print(f"Parsed invariants from clusters: {len(result)}")
print(f"Invariants with ≥1 match: {(result['tx_fail_count'] > 0).sum()}")
print(f"Rows matched to clustered invariants: {covered_rows} ({coverage_pct:.2f}%)")
print(f"Saved:\n- {out_inv}\n- {out_cluster}")

print("\nTop 10 clusters by total failures:")
print(cluster_totals.head(10).to_string(index=False))

print("\nSample of per-invariant counts:")
print(result.sort_values(['cluster_id','tx_fail_count'], ascending=[True,False]).head(15).to_string(index=False))


In [ ]:
# Robust matching of clustered invariants to Parquet dataset.
# Handles:
#   - Cluster bullets like "- inv," and trailing punctuation
#   - require(cond [, "msg"]), assert(cond)
#   - if (cond) revert ... ; and if (!cond) revert ...
#   - { block } or single-statement revert
#   - Simple inversion for if(cond) revert -> invert(cond)
#   - Canonicalization: whitespace, quotes, outer parens, semicolons, and common casts (uint256(...), address(...), payable(...), bytes32(...))
#
# Outputs:
#   - invariant_failure_counts.csv   (cluster_id, invariant, inv_key, tx_fail_count)
#   - cluster_tx_totals.csv          (cluster_id, cluster_tx_fail_total)
#
# Prints coverage stats and a small sample of unmatched invariants.

import re
from pathlib import Path
import pandas as pd

CLUSTERS_TXT = "dbscan_Raven.txt"
PARQUET_FP   = "test_dataset.parquet"

# --------------------- Utilities ---------------------

def strip_redundant_parens(s: str) -> str:
    s = s.strip()
    # repeatedly strip one pair of surrounding parentheses if they match and enclose the whole string
    while s.startswith("(") and s.endswith(")"):
        depth = 0
        ok = True
        for i, ch in enumerate(s):
            if ch == "(":
                depth += 1
            elif ch == ")":
                depth -= 1
                if depth == 0 and i != len(s) - 1:
                    ok = False
                    break
        if ok:
            s = s[1:-1].strip()
        else:
            break
    return s

CAST_PATTERN = re.compile(r"\b(?:u?int(?:8|16|32|64|128|256)?|address|bytes(?:1|2|3|4|5|6|7|8|9|1[0-9]|2[0-9]|3[0-2])?|payable)\s*\(", re.IGNORECASE)

def strip_simple_casts(expr: str) -> str:
    # remove type/address/payable casts by replacing T(expr) -> (expr), repeated until no change
    s = expr
    while True:
        m = CAST_PATTERN.search(s)
        if not m:
            break
        start = m.start()
        # find matching closing parenthesis for the '(' matched
        i = m.end() - 1  # position of '('
        depth = 1
        j = i + 1
        while j < len(s) and depth > 0:
            if s[j] == "(":
                depth += 1
            elif s[j] == ")":
                depth -= 1
            j += 1
        if depth != 0:
            # unbalanced; give up this occurrence
            break
        # replace "Type(" ... ")" with just "(" ... ")"
        s = s[:m.start()] + "(" + s[i+1:j] + s[j:]
    return s

def normalize_key(s: str) -> str:
    if s is None:
        return ""
    s = s.strip()
    # strip leading bullet markers
    s = re.sub(r"^\s*[-–—•]\s*", "", s)
    # strip surrounding quotes/backticks
    s = s.strip('"\'' + "`")
    # remove trailing punctuation/semicolon
    s = s.rstrip(",;. ")
    # remove simple casts and redundant parens early
    s = strip_simple_casts(s)
    s = strip_redundant_parens(s)
    # lowercase and remove all whitespace
    s = s.lower()
    s = re.sub(r"\s+", "", s)
    return s

def find_matching_paren(s: str, open_idx: int) -> int:
    depth = 0
    for i in range(open_idx, len(s)):
        if s[i] == "(":
            depth += 1
        elif s[i] == ")":
            depth -= 1
            if depth == 0:
                return i
    return -1

def split_top_level_comma(s: str) -> tuple[str, str | None]:
    """Split on the first comma not inside parentheses or quotes. Return (left, right_or_None)."""
    depth = 0
    in_str = False
    quote = None
    esc = False
    for i, ch in enumerate(s):
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == quote:
                in_str = False
        else:
            if ch in ("'", '"'):
                in_str = True
                quote = ch
            elif ch == "(":
                depth += 1
            elif ch == ")":
                depth = max(depth - 1, 0)
            elif ch == "," and depth == 0:
                return s[:i].strip(), s[i+1:].strip()
    return s.strip(), None

BIN_OPS = ["<=", ">=", "==", "!=", "<", ">"]
INVERT_OP = {">": "<=", "<": ">=", ">=": "<", "<=": ">", "==": "!=", "!=": "=="}

def invert_simple(expr: str) -> str | None:
    """Invert a simple A op B at top level (outside parens/strings)."""
    s = expr
    depth = 0
    in_str = False
    quote = None
    for i in range(len(s)):
        ch = s[i]
        if in_str:
            if ch == "\\":
                i += 1
                continue
            elif ch == quote:
                in_str = False
            continue
        if ch in ("'", '"'):
            in_str = True
            quote = ch
            continue
        if ch == "(":
            depth += 1
            continue
        if ch == ")":
            depth -= 1
            continue
        if depth == 0:
            # try 2-char ops first
            two = s[i:i+2]
            if two in ("<=", ">=", "==", "!="):
                return s[:i] + INVERT_OP[two] + s[i+2:]
            if ch in ("<", ">"):
                return s[:i] + INVERT_OP[ch] + s[i+1:]
    return None

# --------------------- Parsing invariants from dataset ---------------------

def extract_predicate_variants(full: str) -> list[str]:
    """Return candidate predicate strings for a dataset 'failure_invariant'-like field."""
    if not isinstance(full, str):
        return []
    s = full.strip().rstrip(";")

    out = [s]  # raw as-is

    # require(...) and assert(...)
    for kw in ("require", "assert"):
        idx = s.lower().find(f"{kw}(")
        if idx != -1:
            open_idx = s.find("(", idx)
            close_idx = find_matching_paren(s, open_idx)
            if open_idx != -1 and close_idx != -1:
                inside = s[open_idx+1:close_idx].strip()
                out.append(inside)
                left, _right = split_top_level_comma(inside)
                out.append(left)

    # if (COND) revert ... ;  (single stmt or block)
    if_idx = s.lower().find("if")
    if if_idx != -1:
        # ensure it's 'if' keyword boundary
        if if_idx == 0 or not s[if_idx-1].isalnum():
            open_idx = s.find("(", if_idx)
            close_idx = find_matching_paren(s, open_idx) if open_idx != -1 else -1
            if open_idx != -1 and close_idx != -1:
                cond = s[open_idx+1:close_idx].strip()
                # does following part contain 'revert'?
                tail = s[close_idx+1:]
                if re.search(r"\brevert\b", tail, flags=re.IGNORECASE):
                    out.append(cond)
                    # if the cond is negated like !(...) pull inner
                    m_not = re.match(r"!\s*\((.*)\)\s*$", cond)
                    if m_not:
                        out.append(m_not.group(1).strip())
                    # try inverting simple comparison
                    inv = invert_simple(cond)
                    if inv:
                        out.append(inv)

    # revert Error(!cond)  — capture inner if it is a negation
    m_err = re.search(r"\brevert\s+[A-Za-z_]\w*\s*\((.*)\)", s)
    if m_err:
        inner = m_err.group(1).strip()
        out.append(inner)
        m_not2 = re.match(r"!\s*\((.*)\)\s*$", inner)
        if m_not2:
            out.append(m_not2.group(1).strip())

    # Unique preserve order
    seen = set()
    uniq = []
    for v in out:
        v = v.strip()
        if v and v not in seen:
            seen.add(v)
            uniq.append(v)
    return uniq

# --------------------- Parse cluster file ---------------------

def parse_clusters(path: str) -> pd.DataFrame:
    clusters = []
    current = None
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.rstrip("\n")
            header = re.match(r"^\s*---\s*cluster\s+(\d+)\s*\((\d+)\s*items?\)\s*---\s*$", line, flags=re.IGNORECASE)
            if header:
                current = int(header.group(1))
                continue
            if current is None:
                continue
            if not line.strip() or line.strip().startswith("---"):
                continue
            inv_raw = re.sub(r"^\s*-\s*", "", line).strip().rstrip(",;")
            if not inv_raw:
                continue
            clusters.append({"cluster_id": current, "invariant": inv_raw, "inv_key": normalize_key(inv_raw)})
    df = pd.DataFrame(clusters)
    if df.empty:
        raise ValueError("No invariants parsed from clusters file. Check formatting.")
    return df

# --------------------- Main ---------------------

clusters_df = parse_clusters(CLUSTERS_TXT)

# Load parquet (pandas with pyarrow/fastparquet)
dataset_df = pd.read_parquet(PARQUET_FP)

# Pick invariant column
inv_col = "failure_invariant" if "failure_invariant" in dataset_df.columns else None
if inv_col is None:
    cand = [c for c in dataset_df.columns if any(k in c.lower() for k in ["invariant","predicate"])]
    if not cand:
        raise KeyError(f"'failure_invariant' column not found. Available columns: {list(dataset_df.columns)}")
    inv_col = cand[0]

dataset_df = dataset_df[dataset_df[inv_col].notna()].copy()
dataset_df = dataset_df.reset_index(drop=True).rename(columns={inv_col: "failure_invariant"})
dataset_df["row_id"] = dataset_df.index

# Build candidate keys per row (explode)
candidates = []
for row in dataset_df.itertuples(index=False):
    preds = extract_predicate_variants(getattr(row, "failure_invariant"))
    for p in preds:
        key = normalize_key(p)
        if key:
            candidates.append((row.row_id, key))
cand_df = pd.DataFrame(candidates, columns=["row_id", "inv_key"]).drop_duplicates()

# Counts per candidate key
counts = cand_df.groupby("inv_key")["row_id"].nunique().rename("tx_fail_count")

# Join to cluster invariants
result = clusters_df.merge(counts, how="left", on="inv_key")
result["tx_fail_count"] = result["tx_fail_count"].fillna(0).astype(int)

# Coverage stats
covered_rows = cand_df[cand_df["inv_key"].isin(result["inv_key"])].row_id.nunique()
n_rows = len(dataset_df)
coverage_pct = 100.0 * covered_rows / max(n_rows, 1)

# Cluster totals
cluster_totals = (result.groupby("cluster_id")["tx_fail_count"]
                  .sum()
                  .reset_index()
                  .rename(columns={"tx_fail_count": "cluster_tx_fail_total"})
                  .sort_values("cluster_tx_fail_total", ascending=False))

# Save
out_inv = "invariant_failure_counts.csv"
out_cluster = "cluster_tx_totals.csv"
result.to_csv(out_inv, index=False)
cluster_totals.to_csv(out_cluster, index=False)

# Report
print(f"Dataset rows: {n_rows}")
print(f"Parsed invariants from clusters: {len(result)}")
print(f"Invariants with ≥1 match: {(result['tx_fail_count'] > 0).sum()}")
print(f"Rows matched to clustered invariants: {covered_rows} ({coverage_pct:.2f}%)")
print(f"Saved:\n- {out_inv}\n- {out_cluster}")

# Show a few unmatched for debugging
unmatched = result[result["tx_fail_count"] == 0][["cluster_id","invariant"]].head(15)
if not unmatched.empty:
    print("\nSample unmatched invariants (first 15):")
    print(unmatched.to_string(index=False))

print("\nTop 10 clusters by total failures:")
print(cluster_totals.head(10).to_string(index=False))

print("\nSample of per-invariant counts:")
print(result.sort_values(['cluster_id','tx_fail_count'], ascending=[True,False]).head(15).to_string(index=False))


In [ ]:
# Exact-style extraction to replicate your clustering predicates, then count matches.
# Files: dbscan_Raven.txt, test_dataset.parquet
# Outputs:
#   - invariant_failure_counts.csv  (cluster_id, invariant, tx_fail_count)
#   - cluster_tx_totals.csv         (cluster_id, cluster_tx_fail_total)

import re
from pathlib import Path
import pandas as pd

CLUSTERS_TXT = "dbscan_Raven.txt"
PARQUET_FP   = "test_dataset.parquet"

# -------------------- Helpers that complete your pipeline --------------------

def find_balanced_expr(s: str, open_idx: int) -> str | None:
    """Return the substring inside the parentheses that start at open_idx."""
    if open_idx < 0 or open_idx >= len(s) or s[open_idx] != "(":
        return None
    depth = 0
    for i in range(open_idx, len(s)):
        if s[i] == "(":
            depth += 1
        elif s[i] == ")":
            depth -= 1
            if depth == 0:
                # return inside ( ... )
                return s[open_idx+1:i]
    return None  # unbalanced

def split_condition_and_error(expr: str):
    """Split a require/assert argument into (predicate, error) at a top-level comma."""
    in_quote = False
    escape = False
    depth = 0
    for i, ch in enumerate(expr):
        if ch in ['"', "'"] and not escape:
            in_quote = not in_quote
        elif ch == ',' and not in_quote and depth == 0:
            return expr[:i].strip(), expr[i+1:].strip()
        elif ch == '(':
            depth += 1
        elif ch == ')':
            depth -= 1
        escape = (ch == '\\')
    return expr.strip(), None

def extract_revert_parts(statement: str):
    """
    Try to extract predicate and error from 'if (COND) revert ...' and custom errors.
    Returns (predicate, error_or_None) or (None, None) if no match.
    """
    s = statement.strip()

    # Pattern: if (COND) { revert("msg"); }
    m = re.search(r'if\s*\((.+?)\)\s*\{([^{}]*?)\}', s, flags=re.IGNORECASE | re.DOTALL)
    if m and re.search(r'\brevert\b', m.group(2), flags=re.IGNORECASE):
        cond = m.group(1).strip()
        # Try to pull message inside revert(...)
        m_msg = re.search(r'revert\s*\((.*?)\)\s*;', m.group(2), flags=re.IGNORECASE | re.DOTALL)
        if m_msg:
            err = m_msg.group(1).strip().strip('\'"')
            return cond, err
        # Custom error revert Name(args);
        m_custom = re.search(r'revert\s+[A-Za-z_]\w*\s*\((.*?)\)\s*;', m.group(2), flags=re.IGNORECASE | re.DOTALL)
        if m_custom:
            # We don't know which arg is message; return predicate only
            return cond, None
        # Bare revert;
        if re.search(r'\brevert\s*;', m.group(2), flags=re.IGNORECASE):
            return cond, None

    # Pattern: if (COND) revert("msg");  (single statement)
    m = re.search(r'if\s*\((.+?)\)\s*revert\s*\((.*?)\)\s*;', s, flags=re.IGNORECASE | re.DOTALL)
    if m:
        cond = m.group(1).strip()
        err  = m.group(2).strip().strip('\'"')
        return cond, err

    # Pattern: if (COND) revert CustomError(args...);
    m = re.search(r'if\s*\((.+?)\)\s*revert\s+[A-Za-z_]\w*\s*\((.*?)\)\s*;', s, flags=re.IGNORECASE | re.DOTALL)
    if m:
        cond = m.group(1).strip()
        return cond, None

    # Pattern: if (COND) revert;
    m = re.search(r'if\s*\((.+?)\)\s*revert\s*;', s, flags=re.IGNORECASE)
    if m:
        cond = m.group(1).strip()
        return cond, None

    # Pattern: direct revert("msg"); with a negated arg like revert When(!(cond));
    m = re.search(r'\brevert\s+[A-Za-z_]\w*\s*\((.*?)\)\s*;', s, flags=re.IGNORECASE | re.DOTALL)
    if m:
        inner = m.group(1).strip()
        return inner, None

    return None, None

def extract_parts(statement: str, with_message: bool = True):
    """Your extract_parts logic, completed, closely mirroring your snippet."""
    statement = statement.strip()

    # convert `} else if (...)` to `} if (...)` (drop 'else')
    pattern = r'^(\}\s*)else(\s+if.*)'
    match = re.match(pattern, statement)
    if match:
        statement = match.group(1) + match.group(2)

    # require/assert(...)
    m = re.match(r'^(require|assert)\s*\(', statement, flags=re.IGNORECASE)
    if m:
        open_idx = statement.find('(', m.end() - 1)
        if open_idx != -1:
            inner = find_balanced_expr(statement, open_idx)
            if inner is not None:
                cond, error = split_condition_and_error(inner)
                if error and with_message:
                    error = error.strip().strip('\'"')
                    return cond, error
                return cond, None

    # if {... require(...); }
    m = re.search(r'if\s*\((.+?)\)\s*\{[^}]*?require\s*\((.+?)\);\s*\}', statement, flags=re.IGNORECASE | re.DOTALL)
    if m:
        req_cond, req_error = split_condition_and_error(m.group(2).strip())
        if req_error and with_message:
            req_error = req_error.strip().strip('\'"')
            return req_cond, req_error
        return req_cond, None

    # { require(...); }
    m = re.search(r'\{\s*require\s*\((.+?)\)\s*;', statement, flags=re.IGNORECASE)
    if m:
        cond, error = split_condition_and_error(m.group(1).strip())
        if error and with_message:
            error = error.strip().strip('\'"')
            return cond, error
        return cond, None

    # if (...) require(...);
    m = re.search(r'if\s*\((.+?)\)\s*require\s*\((.+?)\)\s*;', statement, flags=re.IGNORECASE)
    if m:
        req_cond, req_error = split_condition_and_error(m.group(2).strip())
        if req_error and with_message:
            req_error = req_error.strip().strip('\'"')
            return req_cond, req_error
        return req_cond, None

    # bare require(...);
    m = re.match(r'require\s*\((.+?)\)\s*;', statement, flags=re.IGNORECASE)
    if m:
        cond, error = split_condition_and_error(m.group(1))
        if error and with_message:
            error = error.strip().strip('\'"')
            return cond, error
        return cond, None

    # assert foo, "msg"
    m = re.match(r'^assert\s+(.+?)\s*,\s*["\'](.+?)["\'](?:\s*#.*)?$', statement, flags=re.IGNORECASE)
    if m:
        predicate = m.group(1).strip()
        error = m.group(2).strip()
        if error and with_message:
            return predicate, error
        return predicate, None

    # assert foo
    m = re.match(r'^assert\s+(.+)$', statement, flags=re.IGNORECASE)
    if m:
        predicate = m.group(1).strip()
        return predicate, None

    # if (cond) throw;
    m = re.match(r'^if\s*\((.*?)\)\s*throw\s*;', statement.strip(), flags=re.IGNORECASE)
    if m:
        predicate = m.group(1).strip()
        return predicate, None

    # revert-based
    revert_res, error = extract_revert_parts(statement)
    if revert_res and error:
        return revert_res, error
    if revert_res:
        return revert_res, None

    # no match
    return None, None

# -------------------- Your preprocessing replicated --------------------

def preprocessing1(parquet_file_path, with_message=True):
    df = pd.read_parquet(parquet_file_path)

    # filter & lowercase/strip
    df = df[df['failure_invariant'].notna()].copy()
    df['failure_invariant_str'] = df['failure_invariant'].astype(str).str.lower().str.strip()

    assert_require_mask = df['failure_invariant_str'].str.contains(r'\b(assert|require)\b', regex=True, na=False)
    revert_with_if_mask = df['failure_invariant_str'].str.contains(r'\brevert\b', regex=True, na=False) & \
                          df['failure_invariant_str'].str.contains(r'\bif\b', regex=True, na=False)
    throw_with_if_mask  = df['failure_invariant_str'].str.contains(r'\bthrow\b', regex=True, na=False) & \
                          df['failure_invariant_str'].str.contains(r'\bif\b', regex=True, na=False)
    final_mask = assert_require_mask | revert_with_if_mask | throw_with_if_mask
    df = df[final_mask].copy()

    # dedup by the normalized string
    df = df.drop_duplicates(subset='failure_invariant_str').reset_index(drop=True)

    # extract predicate/message
    preds_msgs = df['failure_invariant_str'].apply(lambda s: extract_parts(s, with_message=with_message))
    df[['predicate', 'message']] = pd.DataFrame(preds_msgs.tolist(), index=df.index)
    df = df[df['predicate'].notnull()].copy()

    # prefer failure_reason/failure_message when they are not generic
    df['message'] = None
    if 'failure_reason' in df.columns:
        mask_reason = df['failure_reason'].notna() & ~df['failure_reason'].str.contains('execution reverted', case=False, na=False)
        df.loc[mask_reason, 'message'] = df.loc[mask_reason, 'failure_reason']
    if 'failure_message' in df.columns:
        mask_message = df['failure_message'].notna() & ~df['failure_message'].str.contains('execution reverted', case=False, na=False)
        df.loc[mask_message, 'message'] = df.loc[mask_message, 'failure_message']

    # final unique predicates
    result = df[['predicate', 'message']].drop_duplicates(subset=['predicate']).reset_index(drop=True)
    return result

# -------------------- Build counts over the full 20k --------------------

# Load full dataset (not dedup) to count rows per predicate
full = pd.read_parquet(PARQUET_FP)
full = full[full['failure_invariant'].notna()].copy()
full['failure_invariant_str'] = full['failure_invariant'].astype(str).str.lower().str.strip()

# Use the SAME filtering as your pipeline
assert_require_mask = full['failure_invariant_str'].str.contains(r'\b(assert|require)\b', regex=True, na=False)
revert_with_if_mask = full['failure_invariant_str'].str.contains(r'\brevert\b', regex=True, na=False) & \
                      full['failure_invariant_str'].str.contains(r'\bif\b', regex=True, na=False)
throw_with_if_mask  = full['failure_invariant_str'].str.contains(r'\bthrow\b', regex=True, na=False) & \
                      full['failure_invariant_str'].str.contains(r'\bif\b', regex=True, na=False)
final_mask = assert_require_mask | revert_with_if_mask | throw_with_if_mask
full = full[final_mask].copy()

# Extract predicate for EVERY row (no dedup) so we can count transactions per predicate
extracted = full['failure_invariant_str'].apply(lambda s: extract_parts(s, with_message=True))
full[['predicate', 'message']] = pd.DataFrame(extracted.tolist(), index=full.index)
full = full[full['predicate'].notnull()].copy()

# Count transactions per predicate (canonical form you used)
pred_counts = full.groupby('predicate').size().rename('tx_fail_count')

# -------------------- Parse cluster file into canonical predicate strings --------------------

def parse_clusters_to_predicates(txt_path: str) -> pd.DataFrame:
    """
    From dbscan_Raven.txt, extract:
      cluster_id, invariant (original line), predicate (canonical string you clustered)
    We assume cluster lines are like "- <predicate>," possibly with comments; we strip "- " and trailing ","
    """
    rows = []
    current = None
    with open(txt_path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.rstrip("\n")
            m = re.match(r"^\s*---\s*cluster\s+(\d+)\s*\((\d+)\s*items?\)\s*---\s*$", line, flags=re.IGNORECASE)
            if m:
                current = int(m.group(1))
                continue
            if current is None:
                continue
            if not line.strip() or line.strip().startswith("---"):
                continue
            inv_raw = re.sub(r"^\s*-\s*", "", line)  # strip bullet
            # strip any trailing comment chars and trailing comma/semicolon/space
            inv_raw = re.sub(r"[#].*$", "", inv_raw)  # drop trailing '#' comments if present
            inv_raw = inv_raw.strip().rstrip(",;")
            if not inv_raw:
                continue
            # canonical predicate text exactly as in your pipeline (lower, strip)
            predicate = inv_raw.lower().strip()
            rows.append({"cluster_id": current, "invariant": inv_raw, "predicate": predicate})
    return pd.DataFrame(rows)

clusters_df = parse_clusters_to_predicates(CLUSTERS_TXT)

# -------------------- Join and aggregate --------------------

# Map per-invariant counts by exact canonical predicate key
result = clusters_df.merge(pred_counts, how="left", left_on="predicate", right_index=True)
result['tx_fail_count'] = result['tx_fail_count'].fillna(0).astype(int)

# Cluster totals
cluster_totals = (result.groupby("cluster_id")["tx_fail_count"]
                  .sum()
                  .reset_index()
                  .rename(columns={"tx_fail_count": "cluster_tx_fail_total"})
                  .sort_values("cluster_tx_fail_total", ascending=False))

# Save outputs
out_inv = "invariant_failure_counts.csv"
out_clu = "cluster_tx_totals.csv"
result.to_csv(out_inv, index=False)
cluster_totals.to_csv(out_clu, index=False)

# Report
n_rows = len(pd.read_parquet(PARQUET_FP))
covered_rows = int(full[full['predicate'].isin(result['predicate'])].shape[0])
coverage_pct = 100.0 * covered_rows / max(len(full), 1)

print(f"Dataset rows: {n_rows}")
print(f"Parsed invariants from clusters: {len(result)}")
print(f"Invariants with ≥1 match: {(result['tx_fail_count'] > 0).sum()}")
print(f"Rows matched to clustered invariants: {covered_rows} ({coverage_pct:.2f}%)")
print(f"Saved:\n- {out_inv}\n- {out_clu}")

# Show a few unmatched for quick debugging
unmatched = result[result["tx_fail_count"] == 0][["cluster_id","invariant"]].head(20)
if not unmatched.empty:
    print("\nSample unmatched invariants (first 20):")
    print(unmatched.to_string(index=False))

print("\nTop 10 clusters by total failures:")
print(cluster_totals.head(10).to_string(index=False))

print("\nSample of per-invariant counts:")
print(result.sort_values(['cluster_id','tx_fail_count'], ascending=[True,False]).head(20).to_string(index=False))


In [ ]:
import pandas as pd
import re

# ---------- helpers your code referenced but didn’t define ----------
def find_balanced_expr(s: str, open_idx: int):
    """Return the substring inside the parentheses that start at open_idx, or None if unbalanced."""
    if open_idx < 0 or open_idx >= len(s) or s[open_idx] != "(":
        return None
    depth = 0
    for i in range(open_idx, len(s)):
        if s[i] == "(":
            depth += 1
        elif s[i] == ")":
            depth -= 1
            if depth == 0:
                return s[open_idx+1:i]
    return None

def split_condition_and_error(expr):
    """Split a require/assert argument into (predicate, error) at the first top-level comma."""
    in_quote = False
    escape = False
    depth = 0
    for i, ch in enumerate(expr):
        if ch in ['"', "'"] and not escape:
            in_quote = not in_quote
        elif ch == ',' and not in_quote and depth == 0:
            return expr[:i].strip(), expr[i+1:].strip()
        elif ch == '(':
            depth += 1
        elif ch == ')':
            depth -= 1
        escape = (ch == '\\')
    return expr.strip(), None

def extract_revert_parts(statement: str):
    """
    Extract predicate and (optional) error from revert patterns:
      - if (COND) { revert("msg"); }
      - if (COND) revert("msg");
      - if (COND) revert CustomError(args...);
      - if (COND) revert;
      - revert CustomError(!(cond));
    """
    s = statement.strip()

    # if (COND) { ... revert(...); ... }
    m = re.search(r'if\s*\((.+?)\)\s*\{([^{}]*?)\}', s, flags=re.IGNORECASE | re.DOTALL)
    if m and re.search(r'\brevert\b', m.group(2), flags=re.IGNORECASE):
        cond = m.group(1).strip()
        m_msg = re.search(r'revert\s*\((.*?)\)\s*;', m.group(2), flags=re.IGNORECASE | re.DOTALL)
        if m_msg:
            return cond, m_msg.group(1).strip().strip('\'"')
        m_custom = re.search(r'revert\s+[A-Za-z_]\w*\s*\((.*?)\)\s*;', m.group(2), flags=re.IGNORECASE | re.DOTALL)
        if m_custom:
            return cond, None
        if re.search(r'\brevert\s*;', m.group(2), flags=re.IGNORECASE):
            return cond, None

    # if (COND) revert("msg");
    m = re.search(r'if\s*\((.+?)\)\s*revert\s*\((.*?)\)\s*;', s, flags=re.IGNORECASE | re.DOTALL)
    if m:
        return m.group(1).strip(), m.group(2).strip().strip('\'"')

    # if (COND) revert CustomError(args...);
    m = re.search(r'if\s*\((.+?)\)\s*revert\s+[A-Za-z_]\w*\s*\((.*?)\)\s*;', s, flags=re.IGNORECASE | re.DOTALL)
    if m:
        return m.group(1).strip(), None

    # if (COND) revert;
    m = re.search(r'if\s*\((.+?)\)\s*revert\s*;', s, flags=re.IGNORECASE)
    if m:
        return m.group(1).strip(), None

    # revert CustomError(args...)
    m = re.search(r'\brevert\s+[A-Za-z_]\w*\s*\((.*?)\)\s*;', s, flags=re.IGNORECASE | re.DOTALL)
    if m:
        inner = m.group(1).strip()
        return inner, None

    return None, None

def extract_parts(statement, with_message=True):
    """Exact behavior of your extract_parts (with the missing helpers completed)."""
    statement = statement.strip()

    pattern = r'^(\}\s*)else(\s+if.*)'
    match = re.match(pattern, statement)
    if match:
        statement = match.group(1) + match.group(2)  # drop 'else', keep '} if (...)'

    # require/assert(...)
    m = re.match(r'^(require|assert)\s*\(', statement, flags=re.IGNORECASE)
    if m:
        open_idx = statement.find('(', m.end() - 1)
        if open_idx != -1:
            inner = find_balanced_expr(statement, open_idx)
            if inner:
                cond, error = split_condition_and_error(inner)
                if error and with_message:
                    error = error.strip().strip('\'"')
                    return cond, error
                return cond, None

    # if (...) { ... require(...); ... }
    m = re.search(r'if\s*\((.+?)\)\s*\{[^}]*require\s*\((.+?)\);\s*\}', statement, flags=re.IGNORECASE | re.DOTALL)
    if m:
        req_cond, req_error = split_condition_and_error(m.group(2).strip())
        if req_error and with_message:
            req_error = req_error.strip().strip('\'"')
            return req_cond, req_error
        return req_cond, None

    # { require(...); ... }
    m = re.search(r'\{\s*require\s*\((.+?)\)\s*;', statement, flags=re.IGNORECASE)
    if m:
        cond, error = split_condition_and_error(m.group(1).strip())
        if error and with_message:
            error = error.strip().strip('\'"')
            return cond, error
        return cond, None

    # if (...) require(...);
    m = re.search(r'if\s*\((.+?)\)\s*require\s*\((.+?)\)\s*;', statement, flags=re.IGNORECASE)
    if m:
        req_cond, req_error = split_condition_and_error(m.group(2).strip())
        if req_error and with_message:
            req_error = req_error.strip().strip('\'"')
            return req_cond, req_error
        return req_cond , None

    # require(...);
    m = re.match(r'require\s*\((.+?)\)\s*;', statement, flags=re.IGNORECASE)
    if m:
        cond, error = split_condition_and_error(m.group(1))
        if error and with_message:
            error = error.strip().strip('\'"')
            return cond, error
        return cond, None

    # assert x, "msg"
    m = re.match(r'^assert\s+(.+?)\s*,\s*["\'](.+?)["\'](?:\s*#.*)?$', statement, flags=re.IGNORECASE)
    if m:
        predicate = m.group(1).strip()
        error = m.group(2).strip()
        if error and with_message:
            return predicate, error
        return predicate, None

    # assert x
    m = re.match(r'^assert\s+(.+)$', statement, flags=re.IGNORECASE)
    if m:
        predicate = m.group(1).strip()
        return predicate, None

    # if (cond) throw;
    m = re.match(r'^if\s*\((.*?)\)\s*throw\s*;', statement.strip(), flags=re.IGNORECASE)
    if m:
        predicate = m.group(1).strip()
        return predicate, None

    # revert-based
    revert_res, error = extract_revert_parts(statement)
    if revert_res and error:
        return revert_res, error
    if revert_res:
        return revert_res, None

    # No match (keep behavior: print)
    print("UNMATCHED:", statement)
    return None, None

# ---------- your preprocessing, with a twist: add tx_count per dedup predicate ----------
def preprocessing1_with_counts(parquet_file_path, save_dir="../datasets/master_thesis_20000", with_message=True):
    # 1) Load
    df_all = pd.read_parquet(parquet_file_path)

    # 2) Filter rows that have a failure_invariant
    df_all = df_all[df_all['failure_invariant'].notna()].copy()
    df_all['failure_invariant_str'] = df_all['failure_invariant'].astype(str).str.lower().str.strip()

    # 3) Apply your masks
    assert_require_mask = df_all['failure_invariant_str'].str.contains(r'\b(assert|require)\b', regex=True, na=False)
    revert_with_if_mask = df_all['failure_invariant_str'].str.contains(r'\brevert\b', regex=True, na=False) & \
                          df_all['failure_invariant_str'].str.contains(r'\bif\b', regex=True, na=False)
    throw_with_if_mask  = df_all['failure_invariant_str'].str.contains(r'\bthrow\b', regex=True, na=False) & \
                          df_all['failure_invariant_str'].str.contains(r'\bif\b', regex=True, na=False)
    final_mask = assert_require_mask | revert_with_if_mask | throw_with_if_mask
    df_filtered = df_all[final_mask].copy()

    # 4) For counts: extract predicate for EVERY filtered row (no dedup), then groupby
    pred_msg_full = df_filtered['failure_invariant_str'].apply(lambda s: extract_parts(s, with_message=True))
    df_filtered[['predicate', 'message']] = pd.DataFrame(pred_msg_full.tolist(), index=df_filtered.index)
    df_filtered = df_filtered[df_filtered['predicate'].notnull()].copy()
    pred_counts = df_filtered.groupby('predicate').size().rename('tx_count')  # <-- the twist

    # 5) For your canonical dedup set: drop dup failure_invariant_str, then extract predicate/message
    df = df_filtered.drop_duplicates(subset='failure_invariant_str').reset_index(drop=True).copy()
    preds_msgs = df['failure_invariant_str'].apply(lambda s: extract_parts(s, with_message=with_message))
    df[['predicate', 'message']] = pd.DataFrame(preds_msgs.tolist(), index=df.index)
    df = df[df['predicate'].notnull()].copy()

    # 6) Prefer failure_reason/failure_message when present and non-generic (same as your code)
    df['message'] = None
    if 'failure_reason' in df.columns:
        mask_reason = df['failure_reason'].notna() & ~df['failure_reason'].str.contains('execution reverted', case=False, na=False)
        df.loc[mask_reason, 'message'] = df.loc[mask_reason, 'failure_reason']
    if 'failure_message' in df.columns:
        mask_message = df['failure_message'].notna() & ~df['failure_message'].str.contains('execution reverted', case=False, na=False)
        df.loc[mask_message, 'message'] = df.loc[mask_message, 'failure_message']

    # 7) Final unique predicates (as you did)
    result = df[['predicate', 'message']].drop_duplicates(subset=['predicate']).reset_index(drop=True)

    # 8) Merge in tx_count from the full (non-dedup) side
    result = result.merge(pred_counts, how='left', left_on='predicate', right_index=True)
    result['tx_count'] = result['tx_count'].fillna(0).astype(int)

    # 9) Save like your original, plus a with-counts file
    Path(save_dir).parent.mkdir(parents=True, exist_ok=True)
    result[['predicate', 'message']].to_csv(f"{save_dir}_extracted.csv", index=False, header=True)
    result.to_csv(f"{save_dir}_extracted_with_counts.csv", index=False, header=True)

    return result

# ---------- stringify (unchanged) ----------
def df_to_string(df, with_message=True):
    lines = []
    for _, row in df.iterrows():
        predicate = row['predicate']
        message = row['message']
        if with_message and pd.notnull(message):
            line = f"{predicate}, '{message}'"
        else:
            line = f"{predicate},"
        lines.append(line)
    return lines

# ---------- usage ----------
file_path = "test_dataset.parquet"
result = preprocessing1_with_counts(file_path, with_message=True)

print("Unique predicates:", len(result))
print("With message:", result['message'].notna().sum())
print("Without message:", result['message'].isna().sum())

# Show top by tx_count
print("\nTop 15 predicates by tx_count:")
print(result.sort_values('tx_count', ascending=False).head(15).to_string(index=False))

# Optionally get the comma-suffixed lines (as you used for clustering)
without_message_lines = df_to_string(result[['predicate','message']], with_message=False)
# print('\n'.join(without_message_lines[:20]))


In [ ]:
import pandas as pd
import re

def preprocessing1(parquet_file_path, save_dir = "../datasets/master_thesis_20000", with_message=True):
    df = pd.read_parquet(parquet_file_path)
    df = df[df['failure_invariant'].notna()].copy()
    df['failure_invariant_str'] = df['failure_invariant'].astype(str).str.lower().str.strip()
    assert_require_mask = df['failure_invariant_str'].str.contains(r'\b(assert|require)\b', regex=True, na=False)
    revert_with_if_mask = df['failure_invariant_str'].str.contains(r'\brevert\b', regex=True, na=False) & \
                        df['failure_invariant_str'].str.contains(r'\bif\b', regex=True, na=False)
    throw_with_if_mask = df['failure_invariant_str'].str.contains(r'\bthrow\b', regex=True, na=False) & \
                        df['failure_invariant_str'].str.contains(r'\bif\b', regex=True, na=False)
    final_mask = assert_require_mask | revert_with_if_mask | throw_with_if_mask
    df = df[final_mask].copy()

    # >>> ADDED: keep a pre-dedup copy for counting how many tx map to each predicate
    df_pre_dedup = df.copy()

    df = df.drop_duplicates(subset='failure_invariant_str').reset_index(drop=True)
    unique_invariants = df['failure_invariant_str'].drop_duplicates().reset_index(drop=True)
    print(len(df))
    #print(df['failure_invariant'].to_string(index=False))


    def split_condition_and_error(expr):
        """Split a require/assert argument into (predicate, error)"""
        in_quote = False
        escape = False
        depth = 0
        for i, ch in enumerate(expr):
            if ch in ['"', "'"] and not escape:
                in_quote = not in_quote
            elif ch == ',' and not in_quote and depth == 0:
                return expr[:i].strip(), expr[i+1:].strip()
            elif ch == '(':
                depth += 1
            elif ch == ')':
                depth -= 1
            escape = (ch == '\\')
        return expr.strip(), None


    def extract_parts(statement):
        statement = statement.strip()
        pattern = r'^(\}\s*)else(\s+if.*)'
        match = re.match(pattern, statement)
        if match:
            statement = match.group(1) + match.group(2)  # drop 'else', keep '} if (...)'


        # Handle require(...) and assert(...) with balanced parentheses or inline blocks / if statements
        m = re.match(r'^(require|assert)\s*\(', statement)
        if m:
            open_idx = statement.find('(', m.end() - 1)
            if open_idx != -1:
                inner = find_balanced_expr(statement, open_idx)
                if inner:
                    cond, error = split_condition_and_error(inner)
                    if error and with_message:
                        error = error.strip().strip('\'"')
                        return cond, error
                    return cond, None

        # Handle require inside inline code blocks or after if condition without braces
        m = re.search(r'if\s*\((.+?)\)\s*\{[^}]*require\s*\((.+?)\);\s*\}', statement)
        if m:
            req_cond, req_error = split_condition_and_error(m.group(2).strip())
            if req_error and with_message:
                req_error = req_error.strip().strip('\'"')
                return req_cond, req_error
            return req_cond, None

        m = re.search(r'\{\s*require\s*\((.+?)\);', statement)
        if m:
            cond, error = split_condition_and_error(m.group(1).strip())
            if error and with_message:
                error = error.strip().strip('\'"')
                return cond, error
            return cond, None

        m = re.search(r'if\s*\((.+?)\)\s*require\s*\((.+?)\);', statement)
        if m:
            req_cond, req_error = split_condition_and_error(m.group(2).strip())
            if req_error and with_message:
                req_error = req_error.strip().strip('\'"')
                return req_cond, req_error
            return req_cond , None

        m = re.match(r'require\s*\((.+?)\);', statement)
        if m:
            cond, error = split_condition_and_error(m.group(1))
            if error and with_message:
                error = error.strip().strip('\'"')
                return cond, error
            return cond, None

        # Existing assert matches
        m = re.match(r'^assert\s+(.+?)\s*,\s*["\'](.+?)["\'](?:\s*#.*)?$', statement)
        if m:
            predicate = m.group(1).strip()
            error = m.group(2).strip()
            if error and with_message:
                return predicate, error
            return predicate, None

        m = re.match(r'^assert\s+(.+)$', statement)
        if m:
            predicate = m.group(1).strip()
            return predicate, None

        m = re.match(r'^if\s*\((.*?)\)\s*throw\s*;', statement.strip())
        if m:
            predicate = m.group(1).strip()
            return predicate, None

        # Try revert if/else revert extraction
        revert_res, error = extract_revert_parts(statement)
        if revert_res and error:
            return revert_res, error
        if revert_res:
            return revert_res, None

        # No match
        print("UNMATCHED:", statement)
        return None, None

    df[['predicate', 'message']] = pd.DataFrame(df['failure_invariant_str'].apply(extract_parts).tolist(), index=df.index)
    df = df[df['predicate'].notnull()]
    if 'gas_used' in df.columns and 'gas_price' in df.columns:
        df['gas_loss_eth'] = (df['gas_used'] * df['gas_price']) / 1e18

    df[['message']] = None
    if 'failure_reason' in df.columns:
        mask_reason = df['failure_reason'].notna() & ~df['failure_reason'].str.contains('execution reverted', case=False, na=False)
        df.loc[mask_reason, 'message'] = df.loc[mask_reason, 'failure_reason']

    # If 'failure_reason' doesn't exist or you want to use 'failure_message' as a fallback
    if 'failure_message' in df.columns:
        mask_message = df['failure_message'].notna() & ~df['failure_message'].str.contains('execution reverted', case=False, na=False)
        df.loc[mask_message, 'message'] = df.loc[mask_message, 'failure_message']


    result = df[['predicate', 'message']].drop_duplicates(subset=['predicate']).reset_index(drop=True)

    # >>> ADDED: compute how many original (pre-dedup) rows map to each predicate, then merge
    _all_pred_msg = df_pre_dedup['failure_invariant_str'].apply(extract_parts).tolist()
    df_pre_dedup[['predicate', 'message']] = pd.DataFrame(_all_pred_msg, index=df_pre_dedup.index)
    df_pre_dedup = df_pre_dedup[df_pre_dedup['predicate'].notnull()]
    pred_counts = df_pre_dedup.groupby('predicate').size().rename('tx_count')
    result = result.merge(pred_counts, how='left', left_on='predicate', right_index=True)
    result['tx_count'] = result['tx_count'].fillna(0).astype(int)

    # original output (unchanged schema)
    result[['predicate', 'message']].to_csv(f"{save_dir}_extracted.csv", index=False, header=True)
    # >>> ADDED: optional file with counts included
    result.to_csv(f"{save_dir}_extracted_with_counts.csv", index=False, header=True)

    return result


In [ ]:
file_path = "../datasets/test_dataset.parquet"
df = pd.read_parquet(file_path)

df = preprocessing1(file_path)
print(len(df))

count_without_message = df['message'].isna().sum()
print(count_without_message)
#without_message = df_to_string(df, with_message=False)

In [ ]:
import pandas as pd
import re

def preprocessing1(parquet_file_path, save_dir = "master_thesis_20000", with_message=True):
    df = pd.read_parquet(parquet_file_path)
    df = df[df['failure_invariant'].notna()].copy()
    df['failure_invariant_str'] = df['failure_invariant'].astype(str).str.lower().str.strip()
    assert_require_mask = df['failure_invariant_str'].str.contains(r'\b(assert|require)\b', regex=True, na=False)
    revert_with_if_mask = df['failure_invariant_str'].str.contains(r'\brevert\b', regex=True, na=False) & \
                        df['failure_invariant_str'].str.contains(r'\bif\b', regex=True, na=False)
    throw_with_if_mask = df['failure_invariant_str'].str.contains(r'\bthrow\b', regex=True, na=False) & \
                        df['failure_invariant_str'].str.contains(r'\bif\b', regex=True, na=False)
    final_mask = assert_require_mask | revert_with_if_mask | throw_with_if_mask
    df = df[final_mask].copy()
    df = df.drop_duplicates(subset='failure_invariant_str').reset_index(drop=True)
    unique_invariants = df['failure_invariant_str'].drop_duplicates().reset_index(drop=True)
    print(len(df))
    #print(df['failure_invariant'].to_string(index=False))


    def split_condition_and_error(expr):
        """Split a require/assert argument into (predicate, error)"""
        in_quote = False
        escape = False
        depth = 0
        for i, ch in enumerate(expr):
            if ch in ['"', "'"] and not escape:
                in_quote = not in_quote
            elif ch == ',' and not in_quote and depth == 0:
                return expr[:i].strip(), expr[i+1:].strip()
            elif ch == '(':
                depth += 1
            elif ch == ')':
                depth -= 1
            escape = (ch == '\\')
        return expr.strip(), None


    def extract_parts(statement):
        statement = statement.strip()
        pattern = r'^(\}\s*)else(\s+if.*)'
        match = re.match(pattern, statement)
        if match:
            statement = match.group(1) + match.group(2)  # drop 'else', keep '} if (...)'


        # Handle require(...) and assert(...) with balanced parentheses or inline blocks / if statements
        m = re.match(r'^(require|assert)\s*\(', statement)
        if m:
            open_idx = statement.find('(', m.end() - 1)
            if open_idx != -1:
                inner = find_balanced_expr(statement, open_idx)
                if inner:
                    cond, error = split_condition_and_error(inner)
                    if error and with_message:
                        error = error.strip().strip('\'"')
                        return cond, error
                    return cond, None

        # Handle require inside inline code blocks or after if condition without braces
        m = re.search(r'if\s*\((.+?)\)\s*\{[^}]*require\s*\((.+?)\);\s*\}', statement)
        if m:
            req_cond, req_error = split_condition_and_error(m.group(2).strip())
            if req_error and with_message:
                req_error = req_error.strip().strip('\'"')
                return req_cond, req_error
            return req_cond, None

        m = re.search(r'\{\s*require\s*\((.+?)\);', statement)
        if m:
            cond, error = split_condition_and_error(m.group(1).strip())
            if error and with_message:
                error = error.strip().strip('\'"')
                return cond, error
            return cond, None

        m = re.search(r'if\s*\((.+?)\)\s*require\s*\((.+?)\);', statement)
        if m:
            req_cond, req_error = split_condition_and_error(m.group(2).strip())
            if req_error and with_message:
                req_error = req_error.strip().strip('\'"')
                return req_cond, req_error
            return req_cond , None

        m = re.match(r'require\s*\((.+?)\);', statement)
        if m:
            cond, error = split_condition_and_error(m.group(1))
            if error and with_message:
                error = error.strip().strip('\'"')
                return cond, error
            return cond, None

        # Existing assert matches
        m = re.match(r'^assert\s+(.+?)\s*,\s*["\'](.+?)["\'](?:\s*#.*)?$', statement)
        if m:
            predicate = m.group(1).strip()
            error = m.group(2).strip()
            if error and with_message:
                return predicate, error
            return predicate, None

        m = re.match(r'^assert\s+(.+)$', statement)
        if m:
            predicate = m.group(1).strip()
            return predicate, None

        m = re.match(r'^if\s*\((.*?)\)\s*throw\s*;', statement.strip())
        if m:
            predicate = m.group(1).strip()
            return predicate, None

        # Try revert if/else revert extraction
        revert_res, error = extract_revert_parts(statement)
        if revert_res and error:
            return revert_res, error
        if revert_res:
            return revert_res, None

        # No match
        print("UNMATCHED:", statement)
        return None, None

    df[['predicate', 'message']] = pd.DataFrame(df['failure_invariant_str'].apply(extract_parts).tolist(), index=df.index)
    df = df[df['predicate'].notnull()]
    if 'gas_used' in df.columns and 'gas_price' in df.columns:
        df['gas_loss_eth'] = (df['gas_used'] * df['gas_price']) / 1e18

    df[['message']] = None
    if 'failure_reason' in df.columns:
        mask_reason = df['failure_reason'].notna() & ~df['failure_reason'].str.contains('execution reverted', case=False, na=False)
        df.loc[mask_reason, 'message'] = df.loc[mask_reason, 'failure_reason']

    # If 'failure_reason' doesn't exist or you want to use 'failure_message' as a fallback
    if 'failure_message' in df.columns:
        mask_message = df['failure_message'].notna() & ~df['failure_message'].str.contains('execution reverted', case=False, na=False)
        df.loc[mask_message, 'message'] = df.loc[mask_message, 'failure_message']


    #result = df[['predicate', 'message']].drop_duplicates(subset=['predicate']).reset_index(drop=True)

    result.to_csv(f"{save_dir}_extracted.csv", index=False, header=True)

    

    return result


file_path = "test_dataset.parquet"
df = pd.read_parquet(file_path)

df = preprocessing1(file_path)
print(len(df))
# count_without_message = df['message'].isna().sum()
# print(count_without_message)
# without_message = df_to_string(df, with_message=False)

In [5]:
import re
from pathlib import Path
from typing import Dict

def parse_cluster_file(path: str = "dbscan_Raven.txt") -> Dict[int, Dict[str, int]]:
    """
    Parse a clustering file of the form:
        --- Cluster N (X items) ---
        - invariant_a,
        - invariant_b,
        ...
    into a dict: { N: { "invariant_a": 0, "invariant_b": 0, ... }, ... }.
    """
    clusters: Dict[int, Dict[str, int]] = {}
    current_id = None

    cluster_hdr = re.compile(r"^---\s*Cluster\s+(\d+)\b")
    bullet = re.compile(r"^\s*-\s+(.*)$")

    with Path(path).open("r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue

            # New cluster header
            m = cluster_hdr.match(line)
            if m:
                current_id = int(m.group(1))
                clusters[current_id] = {}
                continue

            # Invariant bullet
            if current_id is not None:
                b = bullet.match(line)
                if b:
                    inv = b.group(1).strip()
                    # drop a single trailing comma if present (keeps commas inside the expression)
                    if inv.endswith(","):
                        inv = inv[:-1].rstrip()
                    if inv:  # ignore accidental empties
                        clusters[current_id][inv] = 0

    return clusters


clusters = parse_cluster_file()
# quick sanity check
total_invariants = sum(len(v) for v in clusters.values())
print(f"Parsed {len(clusters)} clusters with {total_invariants} invariants total.")




Parsed 19 clusters with 424 invariants total.


In [11]:
import pandas as pd

# === INPUT FILE ===
MASTER = "master_thesis_20000_extracted_new.csv"

# === LOAD CSV ===
df = pd.read_csv(MASTER)

# Ensure predicate column exists
assert "predicate" in df.columns, "CSV must contain a 'predicate' column"

# Extract full list of invariants
all_predicates = df["predicate"].astype(str).unique()

# === EXTRACT ALL CLUSTERED KEYS ===
# clusters is your dictionary already in memory
# Example shape: { 0: {"inv1": 0, "inv2": 0}, 18:{...}, ... }

clustered = set()
for cid, d in clusters.items():
    for inv in d.keys():
        clustered.add(inv.strip())

# === FIND UNCLUSTERED ===
# Normalize CSV predicates too (strip spaces)
unclustered = []
for inv in all_predicates:
    if inv.strip() not in clustered:
        unclustered.append(inv.strip())

# Convert to DataFrame
unclustered_df = pd.DataFrame(unclustered, columns=["predicate"])

# === OUTPUT CSV ===
OUT = "unclustered_invariants.csv"
unclustered_df.to_csv(OUT, index=False)

print(f"✅ Done! {len(unclustered_df)} unclustered invariants saved to {OUT}")


✅ Done! 303 unclustered invariants saved to unclustered_invariants.csv


In [14]:
import re
import sys
import pandas as pd
from collections import Counter

OUT_ALL = sys.argv[2] if len(sys.argv) > 2 else "unclustered_compound_invariants.csv"
OUT_BREAKDOWN = sys.argv[3] if len(sys.argv) > 3 else "unclustered_compound_breakdown.csv"

df = pd.read_csv("unclustered_invariants.csv")
if "predicate" not in df.columns:
    raise SystemExit("CSV must contain a 'predicate' column")

# --- Patterns ---
# Comparison operators (order matters: longer first)
cmp_re = re.compile(r"(==|!=|<=|>=|<|>)")

# Logical operators && and ||
logical_double_re = re.compile(r"&&|\|\|")

# Textual booleans 'and' / 'or' as standalone words (case-insensitive)
logical_words_re = re.compile(r"\b(and|or)\b", flags=re.IGNORECASE)

# Bitwise single & or | (ensure not part of && or ||)
bitwise_re = re.compile(r"(?<![&|])([&|])(?![&|])")

# Comma-separated boolean clauses inside parentheses e.g., require(a, b) isn't boolean,
# but constructs like "(a && b, c)" are rare. We'll skip comma heuristics to avoid FPs.

def detect_categories(s: str):
    """Return a set of categories that make this predicate 'compound'."""
    if not isinstance(s, str):
        s = str(s)
    t = s.strip()

    cats = set()
    # Logical connectives
    if logical_double_re.search(t):
        cats.add("logical_&&_||")
    # Textual boolean
    if logical_words_re.search(t):
        cats.add("logical_and_or")
    # Bitwise operators
    if bitwise_re.search(t):
        cats.add("bitwise_&_pipe")
    # Multiple comparisons
    n_cmp = len(cmp_re.findall(t))
    if n_cmp >= 2:
        cats.add("multiple_comparisons")
    return cats

# Apply detection
df["predicate"] = df["predicate"].astype(str)
cats_list = []
is_compound = []
for p in df["predicate"]:
    cats = detect_categories(p)
    cats_list.append(";".join(sorted(cats)) if cats else "")
    is_compound.append(bool(cats))

df["compound_categories"] = cats_list
df["is_compound"] = is_compound

n_total = len(df)
n_compound = int(df["is_compound"].sum())
pct = (n_compound / n_total * 100) if n_total else 0.0

# Save only compound invariants (deduped, keep first occurrence order)
compound_df = df.loc[df["is_compound"], ["predicate", "compound_categories"]].drop_duplicates(subset=["predicate"])
compound_df.to_csv(OUT_ALL, index=False)

# Breakdown: counts per category (multi-label aware)
counter = Counter()
for labels in compound_df["compound_categories"]:
    if labels:
        for lab in labels.split(";"):
            counter[lab] += 1

bd_df = pd.DataFrame(sorted(counter.items()), columns=["category", "count"])
bd_df.to_csv(OUT_BREAKDOWN, index=False)

print(f"Total unclustered invariants: {n_total}")
print(f"Compound invariants:          {n_compound} ({pct:.2f}%)")
print(f"Saved compounds to:           {OUT_ALL}")
print(f"Saved category breakdown to:  {OUT_BREAKDOWN}")
print("\\nTop examples:")
for ex, lab in compound_df.head(10).itertuples(index=False):
    print(f" - {ex}    [{lab}]")

Total unclustered invariants: 303
Compound invariants:          53 (17.49%)
Saved compounds to:           unclustered_compound_invariants.csv
Saved category breakdown to:  unclustered_compound_breakdown.csv
\nTop examples:
 - !((_value != 0) && (allowed[msg.sender][_spender] != 0))    [logical_&&_||;multiple_comparisons]
 - (from != _dex.pair && to != _dex.pair) || ((from == _dex.pair || to == _dex.pair) && _tradingenabled > 0)    [logical_&&_||;multiple_comparisons]
 - from == _owner || to == _owner    [logical_&&_||;multiple_comparisons]
 - launch != 0 && amount <= maxtxamount    [logical_&&_||;multiple_comparisons]
 - uniswapv2pair != from && uniswapv2pair != to    [logical_&&_||;multiple_comparisons]
 - from == address(0) || to == address(0) || allowedfrom[from] || allowedto[to]    [logical_&&_||;multiple_comparisons]
 - zeroforone ? sqrtpricelimitx96 < slot0start.sqrtpricex96 && sqrtpricelimitx96 > tickmath.min_sqrt_ratio : sqrtpricelimitx96 > slot0start.sqrtpricex96 && sqrtpricel

In [3]:
import pandas as pd
# load the file test_dataset_hash_invariant.csv 
df = pd.read_csv("test_dataset_hash_invariant.csv")

In [72]:
import pandas as pd

# Load CSV
df = pd.read_csv("test_dataset_hash_invariant.csv")

# Ensure the failure_invariant column is string
df["failure_invariant"] = df["failure_invariant"].astype(str)

import re

import re

def normalize(s: str) -> str:
    # Convert to string
    if not isinstance(s, str):
        s = str(s)

    # Lowercase
    s = s.lower()

    # Remove ALL whitespace
    s = re.sub(r"\s+", "", s)

    return s


cluster_counts = {}

for cluster_id, invariants in clusters.items():
    print(f"\n=== Cluster {cluster_id} ===")
    cluster_counts[cluster_id] = {}

    for invariant in invariants.keys():
        inv_norm = normalize(invariant)
        count = 0

        for failure in df["failure_invariant"]:
            if inv_norm in normalize(failure):
                count += 1

        cluster_counts[cluster_id][invariant] = count
        print(f"Invariant: {invariant}  → count = {count}")




=== Cluster 0 ===
Invariant: ierc20(path[path.length - 1]).balanceof(to).sub(balancebefore) >= amountoutmin  → count = 259
Invariant: ierc20(tokenin).allowance(msg.sender, address(smartvault)) >= amountin  → count = 10
Invariant: value <= _balanceof(from)  → count = 148
Invariant: amount <= _getfreedbalance(src)  → count = 10
Invariant: amount <= balanceof(sender)  → count = 16
Invariant: balance0adjusted.mul(balance1adjusted) >= uint(_reserve0).mul(_reserve1).mul(1000**2)  → count = 22
Invariant: balanceof(from) >= lockedamount(from) + amount  → count = 3
Invariant: asset.balanceof(to) - prevbalance != amount  → count = 6
Invariant: balance0before.add(uint256(amount0)) <= balance0()  → count = 4
Invariant: basereserve <= uint112(-1) && quotereserve <= uint112(-1)  → count = 5
Invariant: unlockedof(msg.sender) >= amount  → count = 1
Invariant: amount > address(this).balance  → count = 3
Invariant: balanceof(_from) >= _tokens  → count = 1
Invariant: token.allowance(msg.sender, address(

In [ ]:
# get the row 13375 from the df
row_13375 = df.iloc[13375]
row_13375["failure_invariant"]

'require(_canTx[from] || _canTx[to], "");'

In [74]:
"_canTx[from] || _canTx[to]".lower() in row_13375["failure_invariant"].lower()

True

In [4]:
len(df)

20000

In [75]:
cluster_counts

{0: {'ierc20(path[path.length - 1]).balanceof(to).sub(balancebefore) >= amountoutmin': 259,
  'ierc20(tokenin).allowance(msg.sender, address(smartvault)) >= amountin': 10,
  'value <= _balanceof(from)': 148,
  'amount <= _getfreedbalance(src)': 10,
  'amount <= balanceof(sender)': 16,
  'balance0adjusted.mul(balance1adjusted) >= uint(_reserve0).mul(_reserve1).mul(1000**2)': 22,
  'balanceof(from) >= lockedamount(from) + amount': 3,
  'asset.balanceof(to) - prevbalance != amount': 6,
  'balance0before.add(uint256(amount0)) <= balance0()': 4,
  'basereserve <= uint112(-1) && quotereserve <= uint112(-1)': 5,
  'unlockedof(msg.sender) >= amount': 1,
  'amount > address(this).balance': 3,
  'balanceof(_from) >= _tokens': 1,
  'token.allowance(msg.sender, address(this)) >= value': 2,
  '_amount <= unlockedtokensinternal(_from)': 1,
  'super.balanceof(to) + amount <= maxwallet': 1,
  'termrepotoken.balanceof(redeemer) == 0': 1,
  'balanceof(from) >= locked.add(tokenid)': 1,
  'token.balanceof

In [76]:
# add all the counts in cluster_counts together to get total counts per cluster
cluster_totals = {}
for cluster_id, invariants in cluster_counts.items():
    total = sum(invariants.values())
    cluster_totals[cluster_id] = total
cluster_totals


{0: 515,
 1: 3116,
 2: 547,
 3: 450,
 4: 314,
 5: 298,
 6: 43,
 7: 158,
 8: 52,
 9: 56,
 10: 64,
 11: 86,
 12: 58,
 13: 113,
 14: 42,
 15: 82,
 16: 74,
 17: 242,
 18: 648}

In [ ]:
cluster_counts[8]

{'!_validateorderandlisting(order, ordertype.ask, exchange, signature, fees)': 15,
 '!verifyproof(merkleproof, cumulativeamount, msg.sender)': 1,
 '!_checkverifiable(_config, _headerhash, _payloadhash)': 2,
 '!_validateorderandlisting(order, ordertype.bid, exchange, signature, fees)': 5,
 '!checkinvariant(qx, qy, offchainx, offchainy, maximumx, minimumy)': 5,
 'callwithexactgaseveniftargetisnocontract( vc.gaslimit, address(vc.validator), abi.encodewithsignature( "validate(uint256,int256,uint256,int256)", uint256(prevaggregatorroundid), prevaggregatorroundanswer, uint256(_aggregatorroundid), _answer ) )': 1,
 'validateorder(hash, order, sig)': 2,
 'signaturechecker.isvalidsignaturenow( owner, typeddatahash, signature )': 10,
 'outputroot == hashing.hashoutputrootproof(_outputrootproof)': 1,
 'ecdsa.recover( messagehashutils.toethsignedmessagehash( keccak256(abi.encodepacked(msg.sender, action)) ), signature ) == signer': 1,
 '_buildheaderblock(proposer, start, end, roothash)': 1,
 'sign

In [78]:
# let's have the sum of all cluster_totals
sum(cluster_totals.values())

6958

In [65]:
# give a percentage of the sum(cluster_totals.values()) for each cluster total
cluster_percentages = {}
total_sum = sum(cluster_totals.values())
for cluster_id, total in cluster_totals.items():
    percentage = (total / total_sum) * 100 if total_sum > 0 else 0
    cluster_percentages[cluster_id] = percentage
cluster_percentages

{0: 7.40155217016384,
 1: 44.7829836159816,
 2: 7.861454440931302,
 3: 6.4673756826674325,
 4: 4.51279103190572,
 5: 4.282839896521989,
 6: 0.6179936763437769,
 7: 2.270767461914343,
 8: 0.7473411899971256,
 9: 0.8048289738430584,
 10: 0.9198045415349237,
 11: 1.2359873526875538,
 12: 0.8335728657660249,
 13: 1.6240298936475999,
 14: 0.6036217303822937,
 15: 1.178499568841621,
 16: 1.0635240011497558,
 17: 3.4780109226789304,
 18: 9.313020983041104}